# Error Visualization and Analysis of Experimental Data Using Python
## Course: Data Exploration and Visualization (Mini Project - Phase 2)
---
**Author / GitHub:** [DineshMoorthy007](https://github.com/DineshMoorthy007)  
**Repository:** `error-visualization-experimental-data`  
**Current Phase:** Phase 2 — Data Exploration, Preprocessing, Outlier Detection & Error Analysis  
**Technologies:** Python, Jupyter Notebook, Pandas, NumPy, Matplotlib, Seaborn, Git, GitHub

## 1. Aim
To perform systematic data exploration, quality verification, data preprocessing, outlier detection, and experimental error analysis on measurement data from a simple pendulum experiment using Python, quantifying measurement uncertainties against theoretical physical predictions.

## 2. Dataset Description
The dataset comprises **80 experimental observations** of a simple pendulum oscillation across **8 distinct lengths** ($0.20\,\text{m}$ to $1.00\,\text{m}$), with **10 repeated measurement trials** per length configuration.

### Theoretical Physics Model
The ideal time period $T$ of a simple pendulum for small angular displacements is governed by:
$$
T = 2\pi \sqrt{\frac{L}{g}}
$$
where:
- $L$ = Pendulum length in metres ($\text{m}$)
- $T$ = Theoretical time period in seconds ($\text{s}$)
- $g$ = Acceleration due to gravity ($9.81\,\text{m/s}^2$)

### Dataset Attributes (Raw Data)
1. `Experiment_ID`: Unique alphanumeric trial identifier (`EXP001` to `EXP080`).
2. `Trial_Number`: Trial repetition index ($1$ to $10$).
3. `Length_m`: Pendulum length in metres ($0.20, 0.30, \dots, 1.00$).
4. `Theoretical_Period_s`: Mathematically calculated theoretical period in seconds.
5. `Experimental_Period_s`: Observed experimental time period in seconds.

## 3. Import Libraries
We import the foundational scientific and data manipulation libraries required for exploration and statistical analysis.

In [ ]:
import os
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Display settings for tabular outputs
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)
pd.set_option('display.float_format', lambda x: f'{x:.4f}')

print("Required libraries successfully imported!")

## 4. Load Dataset
We load the raw experimental dataset (`pendulum_experimental_data.csv`) using a relative path that works reliably across project directories.

In [ ]:
# Robust path resolution for loading raw dataset
raw_dataset_path = os.path.join('..', 'data', 'pendulum_experimental_data.csv')
if not os.path.exists(raw_dataset_path):
    raw_dataset_path = os.path.join('data', 'pendulum_experimental_data.csv')

df = pd.read_csv(raw_dataset_path)
print(f"Raw dataset successfully loaded from: {raw_dataset_path}")

## 5. Data Exploration
We perform foundational exploratory inspections to examine the dataset structure, head/tail records, dimensionality, and initial summary statistics.

In [ ]:
# A. First 5 records
print("=== First 5 Records (Head) ===")
df.head(5)

In [ ]:
# B. Last 5 records
print("=== Last 5 Records (Tail) ===")
df.tail(5)

In [ ]:
# C. Shape & D. Column Names
rows, cols = df.shape
print(f"Dataset Shape: {df.shape} (Rows: {rows}, Columns: {cols})")
print("\nAttribute Names:")
for i, col in enumerate(df.columns, start=1):
    print(f"  {i}. {col}")

In [ ]:
# E. Data Types & F. Dataset Information
print("=== Dataset Information (df.info()) ===")
df.info()

In [ ]:
# G. Descriptive Summary
print("=== Descriptive Statistical Summary (Raw Data) ===")
df.describe()

### Interpretation — Data Exploration
- **Dimensionality:** The raw dataset contains exactly **80 records** (rows) and **5 attributes** (columns).
- **Observations:** The dataset covers 8 distinct physical pendulum lengths ranging from $0.20\,\text{m}$ to $1.00\,\text{m}$, with 10 repeated trials per length.
- **Range Overview:** Theoretical periods range between $0.8971\,\text{s}$ and $2.0061\,\text{s}$, while experimental periods range from $0.8948\,\text{s}$ to $2.0485\,\text{s}$, confirming close alignment with theoretical physics expectations.

## 6. Data Quality Analysis
We conduct rigorous data quality verification, checking for missing values, duplicate observations, invalid data types, and physical measurement inconsistencies.

In [ ]:
# A. Missing Value Analysis
missing_counts = df.isnull().sum()
missing_pct = (missing_counts / len(df)) * 100.0
missing_table = pd.DataFrame({
    'Missing_Count': missing_counts,
    'Missing_Percentage (%)': missing_pct
})
print("=== Missing Value Analysis ===")
print(missing_table)
print(f"\nTotal Missing Values: {df.isnull().sum().sum()}")

In [ ]:
# B. Duplicate Record Check
duplicate_count = df.duplicated().sum()
print(f"Duplicate Records Count: {duplicate_count}")
if duplicate_count == 0:
    print("Validation Confirmation: No duplicate rows detected in dataset.")

In [ ]:
# C. Physical Validity Validation (Domain Integrity Constraints)
invalid_lengths = df[df['Length_m'] <= 0]
invalid_theo = df[df['Theoretical_Period_s'] <= 0]
invalid_exp = df[df['Experimental_Period_s'] <= 0]

print("=== Physical Constraint Validation ===")
print(f"1. Non-positive Pendulum Lengths (L <= 0)       : {len(invalid_lengths)} invalid records")
print(f"2. Non-positive Theoretical Periods (T_theo <= 0): {len(invalid_theo)} invalid records")
print(f"3. Non-positive Experimental Periods (T_exp <= 0) : {len(invalid_exp)} invalid records")

if len(invalid_lengths) == 0 and len(invalid_theo) == 0 and len(invalid_exp) == 0:
    print("\nValidation Result: All physical measurements are strictly positive and domain-valid.")

### Interpretation — Data Quality Analysis
- **Missing Values:** The dataset contains **0 missing/null values** across all attributes (100% complete).
- **Duplicates:** The dataset contains **0 duplicate records**, ensuring every trial is uniquely identified.
- **Physical Integrity:** All length values ($L > 0$) and time period values ($T > 0$) are strictly positive, satisfying physical boundary requirements for simple pendulum oscillations.

## 7. Data Preprocessing
We perform structured preprocessing: enforcing explicit data types, verifying column precision, and ensuring original measurement units (metres and seconds) remain unmodified for direct scientific interpretation.

In [ ]:
# Explicit type casting and standard precision enforcement
df_preprocessed = df.copy()
df_preprocessed['Experiment_ID'] = df_preprocessed['Experiment_ID'].astype(str)
df_preprocessed['Trial_Number'] = df_preprocessed['Trial_Number'].astype(int)
df_preprocessed['Length_m'] = df_preprocessed['Length_m'].astype(float).round(2)
df_preprocessed['Theoretical_Period_s'] = df_preprocessed['Theoretical_Period_s'].astype(float).round(4)
df_preprocessed['Experimental_Period_s'] = df_preprocessed['Experimental_Period_s'].astype(float).round(4)

print("Preprocessing complete. Data types and precision successfully standardized:")
print(df_preprocessed.dtypes)

### Interpretation — Data Preprocessing
- **Type Enforcement:** Verified that `Trial_Number` is integer, `Length_m` and period columns are float64, and `Experiment_ID` is string.
- **Precision Preservation:** Length measurements are formatted to 2 decimal places and period measurements to 4 decimal places.
- **Preservation of Units:** No artificial scaling or normalization was applied to maintain the physical meaning of measurements in metres (m) and seconds (s).

## 8. Variable Classification
We categorize all dataset variables into their statistical and scientific taxonomy with explicit academic rationales.

In [ ]:
variable_taxonomy = [
    {
        'Variable Name': 'Experiment_ID',
        'Data Type': 'Object (String)',
        'Variable Class': 'Identifier (Nominal)',
        'Role in Experiment': 'Unique key identifying individual observation records (EXP001 to EXP080).'
    },
    {
        'Variable Name': 'Trial_Number',
        'Data Type': 'Integer',
        'Variable Class': 'Numerical (Discrete)',
        'Role in Experiment': 'Repetition index (1 to 10) for trials at a given pendulum length.'
    },
    {
        'Variable Name': 'Length_m',
        'Data Type': 'Float',
        'Variable Class': 'Numerical (Continuous)',
        'Role in Experiment': 'Independent physical variable: length of the pendulum in metres.'
    },
    {
        'Variable Name': 'Theoretical_Period_s',
        'Data Type': 'Float',
        'Variable Class': 'Numerical (Continuous)',
        'Role in Experiment': 'Reference value: ideal oscillation period from T = 2*pi*sqrt(L/g).'
    },
    {
        'Variable Name': 'Experimental_Period_s',
        'Data Type': 'Float',
        'Variable Class': 'Numerical (Continuous)',
        'Role in Experiment': 'Dependent measured variable: observed time period in seconds.'
    }
]

taxonomy_df = pd.DataFrame(variable_taxonomy)
print("=== Variable Classification Table ===")
taxonomy_df

### Interpretation — Variable Classification
- `Length_m` serves as the controlled **independent variable**.
- `Experimental_Period_s` is the **dependent empirical response variable**.
- `Theoretical_Period_s` acts as the **analytical reference standard**.
- `Trial_Number` captures repeated experimental trials under identical controlled conditions.

## 9. Error Calculation
We compute four core experimental discrepancy metrics comparing experimental observations against theoretical reference values:

1. **Error ($\text{Error}_s$):** $\text{Error} = T_{\text{exp}} - T_{\text{theo}}$ (signed deviation)
2. **Absolute Error ($\text{Absolute\_Error}_s$):** $|T_{\text{exp}} - T_{\text{theo}}|$ (error magnitude in seconds)
3. **Relative Error ($\text{Relative\_Error}$):** $\frac{|T_{\text{exp}} - T_{\text{theo}}|}{T_{\text{theo}}}$ (dimensionless ratio)
4. **Percentage Error ($\text{Percentage\_Error}$):** $\text{Relative\_Error} \times 100\%$ (normalized percentage error)

In [ ]:
# Compute experimental error metrics
df_preprocessed['Error_s'] = (
    df_preprocessed['Experimental_Period_s'] - df_preprocessed['Theoretical_Period_s']
).round(4)

df_preprocessed['Absolute_Error_s'] = (
    df_preprocessed['Error_s'].abs()
).round(4)

df_preprocessed['Relative_Error'] = (
    df_preprocessed['Absolute_Error_s'] / df_preprocessed['Theoretical_Period_s']
).round(6)

df_preprocessed['Percentage_Error'] = (
    df_preprocessed['Relative_Error'] * 100.0
).round(4)

print("=== Error Calculation Sample (First 10 Records) ===")
df_preprocessed[['Experiment_ID', 'Length_m', 'Theoretical_Period_s', 'Experimental_Period_s', 'Error_s', 'Absolute_Error_s', 'Percentage_Error']].head(10)

### Interpretation — Error Calculation
- **Signed Error:** Both positive and negative discrepancies occur, representing slight stopwatch trigger timing variances.
- **Magnitude:** The mean absolute error is approximately **$0.0214\,\text{s}$**, which is consistent with typical human reaction time in manual laboratory timing.
- **Relative Precision:** The mean percentage error across all 80 trials is **$1.52\%$**, reflecting high overall measurement fidelity.

## 10. Outlier Detection
We apply the standard **Interquartile Range (IQR)** method to detect statistical anomalies in the experimental error distribution.

### IQR Formulation
- $Q_1 = \text{25th percentile}$
- $Q_3 = \text{75th percentile}$
- $\text{IQR} = Q_3 - Q_1$
- $\text{Lower Bound} = Q_1 - 1.5 \times \text{IQR}$
- $\text{Upper Bound} = Q_3 + 1.5 \times \text{IQR}$

*Note: Outliers in experimental data represent important physical variations (such as timing perturbations or reaction latency) and are flagged with `Outlier_Flag` rather than deleted.*

In [ ]:
# Outlier detection on Percentage_Error using IQR method
q1 = df_preprocessed['Percentage_Error'].quantile(0.25)
q3 = df_preprocessed['Percentage_Error'].quantile(0.75)
iqr = q3 - q1
lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr

# Flag outliers
df_preprocessed['Outlier_Flag'] = (
    (df_preprocessed['Percentage_Error'] < lower_bound) | (df_preprocessed['Percentage_Error'] > upper_bound)
)

outliers_df = df_preprocessed[df_preprocessed['Outlier_Flag']]

print("=== IQR Outlier Analysis Summary ===")
print(f"First Quartile (Q1)     : {q1:.4f} %")
print(f"Third Quartile (Q3)     : {q3:.4f} %")
print(f"Interquartile Range(IQR): {iqr:.4f} %")
print(f"Lower Outlier Threshold : {lower_bound:.4f} %")
print(f"Upper Outlier Threshold : {upper_bound:.4f} %")
print(f"Potential Outliers Count: {len(outliers_df)} out of {len(df_preprocessed)} records ({len(outliers_df)/len(df_preprocessed)*100:.2f}%)")
print(f"Outlier Experiment IDs  : {outliers_df['Experiment_ID'].tolist()}")

print("\n=== Outlier Records Detail ===")
outliers_df[['Experiment_ID', 'Length_m', 'Theoretical_Period_s', 'Experimental_Period_s', 'Percentage_Error', 'Outlier_Flag']]

### Interpretation — Outlier Detection
- **Identified Outliers:** Exactly **5 potential outliers** were detected based on the $1.5 \times \text{IQR}$ threshold on percentage error (`EXP004`, `EXP007`, `EXP014`, `EXP038`, `EXP055`).
- **Diagnostic Value:** These observations correspond to trials with higher reaction delays or stopwatch pressing offsets ($> 4.00\%$ error), providing valuable test cases for residual and dispersion analysis.
- **Retention Policy:** Outliers are retained in the dataset with `Outlier_Flag = True` to preserve empirical transparency and prevent data distortion.

## 11. Error Classification
To support qualitative error analysis and future categorical visualizations, we categorize trials into project-defined analytical error quality tiers:
- **Low Error (< 1%):** High-precision measurements with minimal timing error.
- **Moderate Error (1% to 2%):** Acceptable laboratory measurement variance.
- **High Error (>= 2%):** Noticeable timing offsets or experimental perturbations.

In [ ]:
# Discretize into error quality categories
conditions = [
    df_preprocessed['Percentage_Error'] < 1.0,
    (df_preprocessed['Percentage_Error'] >= 1.0) & (df_preprocessed['Percentage_Error'] < 2.0),
    df_preprocessed['Percentage_Error'] >= 2.0
]
categories = ['Low Error (<1%)', 'Moderate Error (1-2%)', 'High Error (>=2%)']

df_preprocessed['Error_Category'] = np.select(conditions, categories, default='Unclassified')

# Summary distribution
category_counts = df_preprocessed['Error_Category'].value_counts().reindex(categories, fill_value=0)
category_pct = (category_counts / len(df_preprocessed)) * 100.0

category_summary_df = pd.DataFrame({
    'Error_Category': category_counts.index,
    'Observation_Count': category_counts.values,
    'Percentage_Share (%)': category_pct.values
})

print("=== Error Category Distribution ===")
category_summary_df

### Interpretation — Error Classification
- **Low Error (< 1%):** Accounts for **39 observations (48.75%)** of the dataset, demonstrating high experimental repeatability.
- **Moderate Error (1% to 2%):** Accounts for **23 observations (28.75%)**.
- **High Error (>= 2%):** Accounts for **18 observations (22.50%)**, which includes the 5 IQR-flagged outliers.
- **Cumulative Performance:** **77.50%** of all observations exhibit measurement errors strictly under $2.0\%$.

## 12. Descriptive Statistical Analysis
We calculate the full suite of descriptive statistical metrics required for the college mini project lab record across key experimental variables.

In [ ]:
# Comprehensive descriptive statistical calculations
target_variables = ['Experimental_Period_s', 'Absolute_Error_s', 'Percentage_Error']
stats_rows = []

for col in target_variables:
    s = df_preprocessed[col]
    mean_val = s.mean()
    median_val = s.median()
    min_val = s.min()
    max_val = s.max()
    range_val = max_val - min_val
    var_val = s.var()
    std_val = s.std()
    q1_val = s.quantile(0.25)
    q2_val = s.quantile(0.50)
    q3_val = s.quantile(0.75)
    
    # Continuous mode note
    rounded_mode = s.round(2).mode()
    mode_str = f"{rounded_mode.iloc[0]:.2f} (rounded)" if not rounded_mode.empty else "N/A"
    
    stats_rows.append({
        'Metric_Variable': col,
        'Mean': mean_val,
        'Median (Q2)': median_val,
        'Mode': mode_str,
        'Minimum': min_val,
        'Maximum': max_val,
        'Range': range_val,
        'Variance': var_val,
        'Std_Deviation': std_val,
        'Q1 (25th %)': q1_val,
        'Q2 (50th %)': q2_val,
        'Q3 (75th %)': q3_val
    })

stats_summary_table = pd.DataFrame(stats_rows)
print("=== Comprehensive Descriptive Statistics Table ===")
stats_summary_table

### Interpretation — Descriptive Statistics
- **Central Tendency:** Mean experimental period is **$1.4704\,\text{s}$** with a median of **$1.4952\,\text{s}$**, matching closely across the multi-length distribution.
- **Error Dispersion:** The mean absolute error is **$0.0214\,\text{s}$** with a low standard deviation of **$0.0267\,\text{s}$**, confirming tight experimental clustering.
- **Percentage Error Distribution:** The median percentage error is **$1.0119\%$**, while the 75th percentile ($Q_3$) is **$1.8374\%$**, confirming that half of all measurements have under $\approx 1\%$ error.

## 13. Length-wise Summary
We group observations by pendulum length (`Length_m`) to evaluate experimental precision, variability, and maximum percentage error across different physical configurations.

In [ ]:
# Grouped summary by pendulum length
length_summary = df_preprocessed.groupby('Length_m').agg(
    Observation_Count=('Experiment_ID', 'count'),
    Mean_Exp_Period_s=('Experimental_Period_s', 'mean'),
    Mean_Theo_Period_s=('Theoretical_Period_s', 'first'),
    Mean_Absolute_Error_s=('Absolute_Error_s', 'mean'),
    Mean_Percentage_Error=('Percentage_Error', 'mean'),
    Std_Exp_Period_s=('Experimental_Period_s', 'std'),
    Max_Percentage_Error=('Percentage_Error', 'max')
).reset_index()

print("=== Length-wise Aggregation Summary ===")
length_summary

### Interpretation — Length-wise Summary
- **Length Scaling:** Mean experimental period steadily increases from **$0.9150\,\text{s}$** ($L = 0.20\,\text{m}$) to **$2.0012\,\text{s}$** ($L = 1.00\,\text{m}$), demonstrating non-linear square-root scaling ($T \propto \sqrt{L}$).
- **Standard Deviation Stability:** Across all 8 length groups, the standard deviation of experimental period remains consistently small (between $0.0159\,\text{s}$ and $0.0533\,\text{s}$).
- **Relative Error Stability:** Mean percentage error per length group stays reliably between $0.91\%$ and $2.10\%$, confirming stable experimental precision across all pendulum dimensions.

## 14. Save Processed Dataset
We export the processed, error-augmented dataset to `data/processed/pendulum_processed_data.csv` while ensuring the original raw CSV remains intact.

In [ ]:
# Create processed data directory and save dataset
processed_dir = os.path.join('..', 'data', 'processed')
if not os.path.exists(os.path.join('..', 'data')):
    processed_dir = os.path.join('data', 'processed')

os.makedirs(processed_dir, exist_ok=True)
processed_file_path = os.path.join(processed_dir, 'pendulum_processed_data.csv')

df_preprocessed.to_csv(processed_file_path, index=False)
print(f"Processed dataset successfully saved to: {processed_file_path}")
print(f"Processed Dataset Shape: {df_preprocessed.shape} (80 records, {df_preprocessed.shape[1]} attributes)")